In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@pravdatalake.dfs.core.windows.net"
GOLD_TABLE_PATH = f"{GOLD_PATH}/fact_ad_listing"
GOLD_TABLE_NAME = "vehicle_sales.gold.fact_ad_listing"

In [0]:
silver_ad = spark.read.format("delta").load(f"{SILVER_PATH}/ad_table")

In [0]:
dim_genmodel = spark.read.format("delta").load(f"{GOLD_PATH}/dim_genmodel")

In [0]:
dimension_date = spark.read.format("delta").load(f"{GOLD_PATH}/dimension_date")

In [0]:
dim_genmodel_lookup = dim_genmodel.select("Genmodel_ID")

In [0]:
dim_date_lookup = (
        dimension_date
        .groupBy("year", "month")
        .agg(min("date_key").alias("date_key"))
    )

In [0]:
fact_ad_listing_updates = (
    silver_ad
    .join(
        dim_date_lookup,
        (silver_ad["Adv_year"] == dim_date_lookup["year"]) & (silver_ad["Adv_month"] == dim_date_lookup["month"]),
        how="left"
    )
    .withColumn("gold_updated_timestamp", current_timestamp())
    .select(
        "Adv_ID", "Genmodel_ID", "date_key",
        "Color", "Reg_year", "Bodytype", "Runned_Miles", "Engine_size_l",
        "Gearbox", "Fuel_type", "Price", "Seat_num", "Door_num", "gold_updated_timestamp"
    )
)

In [0]:
fact_ad_listing_updates.display()

####Data Quality checks

In [0]:
row_count = fact_ad_listing_updates.count()

In [0]:
duplicate_key_count = fact_ad_listing_updates.groupBy("Adv_ID").count().filter("count > 1").count()

In [0]:
unresolved_genmodel_count = (
    fact_ad_listing_updates.join(dim_genmodel_lookup, "Genmodel_ID", "left_anti").count()
)

In [0]:
print(f"row count: {row_count}")
print(f"duplicate Adv_ID count: {duplicate_key_count}")
print(f"Genmodel_ID not found in dim_genmodel count: {unresolved_genmodel_count}")

In [0]:
#assert duplicate_key_count == 0, "Adv_ID should be unique in fact_ad_listing"

In [0]:
if DeltaTable.isDeltaTable(spark, GOLD_TABLE_PATH):
 
    fact_ad_listing_table = DeltaTable.forPath(spark, GOLD_TABLE_PATH)
 
    (fact_ad_listing_table.alias("t")
        .merge(fact_ad_listing_updates.alias("s"), "t.Adv_ID = s.Adv_ID")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    fact_ad_listing_updates.write \
        .format("delta") \
        .mode("overwrite") \
        .save(GOLD_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_TABLE_PATH}'
""")

In [0]:

spark.sql(f"OPTIMIZE {GOLD_TABLE_NAME} ZORDER BY (Genmodel_ID, date_key)")